# Checkpoint 4 — Mock Contest 4 Solutions

Reference solutions for all 7 problems. Each is a complete stdin/stdout program; the cells below are display-only mirrors of the solver files in the `assets/` folder.

## Question 1

**Notice:** scan the grid and recursively flood-fill each not-yet-visited `#` region with a `visited` set, counting the regions.

In [ ]:
import sys

data = sys.stdin.read()
lines = data.split("\n")
header = lines[0].split()
rows = int(header[0])
cols = int(header[1])
grid = []
r = 0
while r < rows:
    grid.append(lines[1 + r])
    r = r + 1
visited = set()

def fill(cr, cc):
    visited.add((cr, cc))
    neighbours = [(cr - 1, cc), (cr + 1, cc), (cr, cc - 1), (cr, cc + 1)]
    for spot in neighbours:
        nr = spot[0]
        nc = spot[1]
        if 0 <= nr and nr < rows and 0 <= nc and nc < cols:
            if grid[nr][nc] == "#" and (nr, nc) not in visited:
                fill(nr, nc)

count = 0
r = 0
while r < rows:
    c = 0
    while c < cols:
        if grid[r][c] == "#" and (r, c) not in visited:
            count = count + 1
            fill(r, c)
        c = c + 1
    r = r + 1
print(str(count))


## Question 2

**Notice:** BFS from `S` with a `deque` FIFO queue and a `visited` set, printing the fewest steps to `T` (or `-1` if unreachable).

In [ ]:
from collections import deque
import sys

data = sys.stdin.read()
lines = data.split("\n")
header = lines[0].split()
rows = int(header[0])
cols = int(header[1])
grid = []
r = 0
while r < rows:
    grid.append(lines[1 + r])
    r = r + 1
start = (0, 0)
target = (0, 0)
r = 0
while r < rows:
    c = 0
    while c < cols:
        if grid[r][c] == "S":
            start = (r, c)
        if grid[r][c] == "T":
            target = (r, c)
        c = c + 1
    r = r + 1
queue = deque()
queue.append((start[0], start[1], 0))
visited = set()
visited.add(start)
answer = "-1"
while len(queue) > 0 and answer == "-1":
    item = queue.popleft()
    cr = item[0]
    cc = item[1]
    dist = item[2]
    if (cr, cc) == target:
        answer = str(dist)
    else:
        neighbours = [(cr - 1, cc), (cr + 1, cc), (cr, cc - 1), (cr, cc + 1)]
        for spot in neighbours:
            nr = spot[0]
            nc = spot[1]
            if 0 <= nr and nr < rows and 0 <= nc and nc < cols:
                if grid[nr][nc] != "#" and (nr, nc) not in visited:
                    visited.add((nr, nc))
                    queue.append((nr, nc, dist + 1))
print(answer)


## Question 3

**Notice:** DFS from node 1 marking a `visited` set, then print `YES` if all `n` nodes were reached, otherwise `NO`.

In [ ]:
import sys

data = sys.stdin.read()
lines = data.split("\n")
header = lines[0].split()
n = int(header[0])
m = int(header[1])
adj = {}
node = 1
while node <= n:
    adj[node] = []
    node = node + 1
i = 0
while i < m:
    parts = lines[1 + i].split()
    u = int(parts[0])
    v = int(parts[1])
    adj[u].append(v)
    adj[v].append(u)
    i = i + 1
visited = set()

def walk(u, seen):
    seen.add(u)
    for nb in adj[u]:
        if nb not in seen:
            walk(nb, seen)

walk(1, visited)
if len(visited) == n:
    print("YES")
else:
    print("NO")


## Question 4

**Notice:** converging two pointers: record `min(h[lo], h[hi]) * (hi - lo)`, then move the pointer at the SHORTER post inward (move `hi` on a tie). Every trough that keeps the shorter post and uses a closer partner is no taller and narrower, so it cannot hold more water; dropping that post loses nothing. Each step moves one pointer, so the scan takes `N - 1` steps instead of checking all `N^2 / 2` pairs.

In [ ]:
import sys

data = sys.stdin.read()
tokens = data.split()
n = int(tokens[0])
heights = []
i = 0
while i < n:
    heights.append(int(tokens[1 + i]))
    i = i + 1
lo = 0
hi = n - 1
best = 0
while lo < hi:
    water = min(heights[lo], heights[hi]) * (hi - lo)
    if water > best:
        best = water
    if heights[lo] < heights[hi]:
        lo = lo + 1
    else:
        hi = hi - 1
print(str(best))


## Question 5

**Notice:** sliding window: add `days[right]` to the window, counting it if it is rainy (`0`). While the count is larger than `K`, move `left` forward, lowering the count whenever a rainy day leaves the window. After that the window is allowed, so compare `right - left + 1` against the best length. Each day enters and leaves the window at most once, so the whole scan is linear in `N`.

In [ ]:
import sys

data = sys.stdin.read()
tokens = data.split()
n = int(tokens[0])
allowed = int(tokens[1])
days = []
i = 0
while i < n:
    days.append(int(tokens[2 + i]))
    i = i + 1
left = 0
rainy = 0
best = 0
right = 0
while right < n:
    if days[right] == 0:
        rainy = rainy + 1
    while rainy > allowed:
        if days[left] == 0:
            rainy = rainy - 1
        left = left + 1
    span = right - left + 1
    if span > best:
        best = span
    right = right + 1
print(str(best))


## Question 6

**Notice:** build `modulus = 10^D` by multiplying by 10 `D` times. Then use repeated squaring: while `exponent > 0`, multiply the answer by `current` when the exponent is odd, square `current`, and halve the exponent, reducing modulo `modulus` after every multiplication. Starting from `1 % modulus` makes `E = 0` give `1`. About 60 halvings handle `E` up to `10^18`, and the reductions keep every number below `10^36`. Finally, pad with leading `0` characters to exactly `D` digits.

In [ ]:
import sys

data = sys.stdin.read()
tokens = data.split()
base = int(tokens[0])
exponent = int(tokens[1])
digits = int(tokens[2])
modulus = 1
i = 0
while i < digits:
    modulus = modulus * 10
    i = i + 1
answer = 1 % modulus
current = base % modulus
while exponent > 0:
    if exponent % 2 == 1:
        answer = answer * current % modulus
    current = current * current % modulus
    exponent = exponent // 2
text = str(answer)
while len(text) < digits:
    text = "0" + text
print(text)


## Question 7

**Notice:** BFS from `start` over a DIRECTED adjacency list (each edge added one way only), printing the visit order.

In [ ]:
from collections import deque
import sys

data = sys.stdin.read()
lines = data.split("\n")
header = lines[0].split()
n = int(header[0])
m = int(header[1])
start = int(header[2])
adj = {}
node = 1
while node <= n:
    adj[node] = []
    node = node + 1
i = 0
while i < m:
    parts = lines[1 + i].split()
    u = int(parts[0])
    v = int(parts[1])
    adj[u].append(v)
    i = i + 1
visited = {start}
queue = deque()
queue.append(start)
order = ""
while len(queue) > 0:
    u = queue.popleft()
    if len(order) > 0:
        order = order + " "
    order = order + str(u)
    for nb in adj[u]:
        if nb not in visited:
            visited.add(nb)
            queue.append(nb)
print(order)
